In [1]:
from web3 import Web3

ganache_url = "http://127.0.0.1:7545"
web3 = Web3(Web3.HTTPProvider(ganache_url))

if web3.is_connected():
    print("Connected to Ganache successfully!")
else:
    print("Connection failed.")

Connected to Ganache successfully!


In [2]:
import pandas as pd

df = pd.read_csv("formatted_healthcare_data.csv")

print(df.head())
print("Total CSV records:", len(df))

                    timestamp device_id       data_type   data_value
0  2026-05-23 22:55:20.319812    PAT440    Oxygen Level          99%
1  2026-05-23 15:18:20.319812    PAT434      Heart Rate       93 bpm
2  2026-05-24 01:06:20.319812    PAT355      Heart Rate       79 bpm
3  2026-05-23 19:19:20.319812    PAT134    Oxygen Level          96%
4  2026-05-24 00:08:20.319812    PAT922  Blood Pressure  127/85 mmHg
Total CSV records: 100


In [3]:
import json

contract_address = "0x8e1c7A22833Fe4699B7fba08754eE315d87731f4"

abi = json.loads("""
[
	{
		"inputs": [],
		"stateMutability": "nonpayable",
		"type": "constructor"
	},
	{
		"anonymous": false,
		"inputs": [
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "deviceId",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "dataType",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "dataValue",
				"type": "string"
			}
		],
		"name": "DataStored",
		"type": "event"
	},
	{
		"inputs": [
			{
				"internalType": "string",
				"name": "_deviceId",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "_dataType",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "_dataValue",
				"type": "string"
			}
		],
		"name": "storeData",
		"outputs": [],
		"stateMutability": "nonpayable",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			}
		],
		"name": "dataRecords",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			},
			{
				"internalType": "string",
				"name": "deviceId",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "dataType",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "dataValue",
				"type": "string"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "index",
				"type": "uint256"
			}
		],
		"name": "getRecord",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			},
			{
				"internalType": "string",
				"name": "",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "",
				"type": "string"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [],
		"name": "getTotalRecords",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [],
		"name": "MAX_ENTRIES",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [],
		"name": "owner",
		"outputs": [
			{
				"internalType": "address",
				"name": "",
				"type": "address"
			}
		],
		"stateMutability": "view",
		"type": "function"
	}
]
""")

contract = web3.eth.contract(
    address=contract_address,
    abi=abi
)

web3.eth.default_account = web3.eth.accounts[0]

print("Smart contract connected successfully!")

Smart contract connected successfully!


In [4]:
before_count = contract.functions.getTotalRecords().call()
print("Records before upload:", before_count)

Records before upload: 2


In [5]:
for index, row in df.head(98).iterrows():

    txn = contract.functions.storeData(
        str(row["device_id"]),
        str(row["data_type"]),
        str(row["data_value"])
    ).transact({
        "from": web3.eth.default_account,
        "gas": 3000000
    })

    web3.eth.wait_for_transaction_receipt(txn)

    print(f"Uploaded record {index + 1}")

Uploaded record 1
Uploaded record 2
Uploaded record 3
Uploaded record 4
Uploaded record 5
Uploaded record 6
Uploaded record 7
Uploaded record 8
Uploaded record 9
Uploaded record 10
Uploaded record 11
Uploaded record 12
Uploaded record 13
Uploaded record 14
Uploaded record 15
Uploaded record 16
Uploaded record 17
Uploaded record 18
Uploaded record 19
Uploaded record 20
Uploaded record 21
Uploaded record 22
Uploaded record 23
Uploaded record 24
Uploaded record 25
Uploaded record 26
Uploaded record 27
Uploaded record 28
Uploaded record 29
Uploaded record 30
Uploaded record 31
Uploaded record 32
Uploaded record 33
Uploaded record 34
Uploaded record 35
Uploaded record 36
Uploaded record 37
Uploaded record 38
Uploaded record 39
Uploaded record 40
Uploaded record 41
Uploaded record 42
Uploaded record 43
Uploaded record 44
Uploaded record 45
Uploaded record 46
Uploaded record 47
Uploaded record 48
Uploaded record 49
Uploaded record 50
Uploaded record 51
Uploaded record 52
Uploaded record 53
Up

In [6]:
total_records = contract.functions.getTotalRecords().call()
print("Total records now stored on blockchain:", total_records)

Total records now stored on blockchain: 100


In [7]:
record = contract.functions.getRecord(2).call()
print("First CSV record from blockchain:", record)

First CSV record from blockchain: [1779613423, 'PAT440', 'Oxygen Level', '99%']
